# Climate Indices with Earthkit & Xclim

This notebook demonstrates how to compute and visualize **climate indices** from CMIP6 datasets using the `earthkit-climate` and `xclim` packages.

We'll use:
- **Precipitation-based indices**:
  - *SDII*: Simple Daily Intensity Index (average precipitation on wet days)
  - *CWD*: Consecutive Wet Days (max number of wet days in a row)

- **Temperature-based indices**:
  - *DTR*: Daily Temperature Range (Tmax - Tmin)
  - *WSDI*: Warm Spell Duration Index (≥6 consecutive days above 90th percentile)
  - *HDD*: Heating Degree Days (based on temperature below threshold)

We’ll load **ACCESS-CM2 CMIP6 data** for both *historical* and *SSP585* scenarios.


In [1]:
import xarray as xr
import earthkit.data as ekd
import earthkit.plots as ekp
from earthkit.climate.indicators.temperature import (
    daily_temperature_range,
    heating_degree_days,
    warm_spell_duration_index,
)
from earthkit.climate.utils.percentile import calculate_percentile_doy

ekd.settings.set("cache-policy", "user")


## Loading CMIP6 data

We’ll use *daily gridded data* from the ACCESS-CM2 model for precipitation (`pr`), maximum (`tasmax`) and minimum (`tasmin`) temperature, for both historical and SSP585 future scenarios.


In [2]:
# Load precipitation
pr_hist = ekd.from_source(
    "url",
    "https://sites.ecmwf.int/repository/earthkit-climate/pr_gridded_day_CMIP6_ACCESS-CM2_r1i1p1f1_deepESD_day_historical.nc",
)
pr_ssp585 = ekd.from_source(
    "url",
    "https://sites.ecmwf.int/repository/earthkit-climate/pr_gridded_day_CMIP6_ACCESS-CM2_r1i1p1f1_deepESD_day_ssp585.nc",
)

# Load temperature
tasmin_hist = ekd.from_source(
    "url",
    "https://sites.ecmwf.int/repository/earthkit-climate/tasmin_gridded_day_CMIP6_ACCESS-CM2_r1i1p1f1_deepESD_day_historical.nc",
)
tasmin_ssp585 = ekd.from_source(
    "url",
    "https://sites.ecmwf.int/repository/earthkit-climate/tasmin_gridded_day_CMIP6_ACCESS-CM2_r1i1p1f1_deepESD_day_ssp585.nc",
)

tasmax_hist = ekd.from_source(
    "url",
    "https://sites.ecmwf.int/repository/earthkit-climate/tasmax_gridded_day_CMIP6_ACCESS-CM2_r1i1p1f1_deepESD_day_historical.nc",
)
tasmax_ssp585 = ekd.from_source(
    "url",
    "https://sites.ecmwf.int/repository/earthkit-climate/tasmax_gridded_day_CMIP6_ACCESS-CM2_r1i1p1f1_deepESD_day_ssp585.nc",
)

In [8]:
tasmax_hist_k = tasmax_hist.to_xarray() + 273.15
tasmax_hist_k["tasmax"].attrs["units"] = "K"
tasmin_hist_k = tasmin_hist.to_xarray() + 273.15
tasmin_hist_k["tasmin"].attrs["units"] = "K"

ds_hist_k = xr.merge([tasmax_hist_k, tasmin_hist_k])
ds_hist_k

/var/folders/l2/529q7bzs665bnrn7_wjx1nsr0000gn/T/ipykernel_90896/4161819160.py:6: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds_hist_k = xr.merge([tasmax_hist_k, tasmin_hist_k])


<xarray.Dataset> Size: 236MB
Dimensions:  (lat: 48, lon: 84, time: 7305)
Coordinates:
  * lat      (lat) float64 384B 41.62 41.67 41.72 41.77 ... 43.87 43.92 43.97
  * lon      (lon) float64 672B -9.575 -9.525 -9.475 ... -5.525 -5.475 -5.425
  * time     (time) datetime64[ns] 58kB 1995-01-01 1995-01-02 ... 2014-12-31
    height   float64 8B 2.0
Data variables:
    tasmax   (time, lat, lon) float32 118MB dask.array<chunksize=(3653, 24, 42), meta=np.ndarray>
    tasmin   (time, lat, lon) float32 118MB dask.array<chunksize=(3653, 24, 42), meta=np.ndarray>
Attributes:
    model:         ACCESS-CM2_r1i1p1f1_deepESD
    scenario:      historical
    project_name:  cmip6
    project_type:  projections

In [9]:
# DTR
dtr = daily_temperature_range(ds_hist_k)


Before convert_units: 279.80032
Before convert_units: K
After convert_units: 6.667464733123779
After convert_units: degC
ek-clim wrapper Args: ()
ek-clim wrapper Kwargs: {}


/Users/edwardcomyn-platt/Work/Git_Repositories/EARTHKIT/earthkit-transforms/.conda/lib/python3.12/site-packages/xclim/core/cfchecks.py:77: UserWarning: Variable does not have a `cell_methods` attribute.
  _check_cell_methods(getattr(vardata, "cell_methods", None), data["cell_methods"])
/Users/edwardcomyn-platt/Work/Git_Repositories/EARTHKIT/earthkit-transforms/.conda/lib/python3.12/site-packages/xclim/core/cfchecks.py:79: UserWarning: Variable does not have a `standard_name` attribute.
  check_valid(vardata, "standard_name", data["standard_name"])


In [10]:
dtr.to_xarray()

<xarray.Dataset> Size: 324kB
Dimensions:  (lat: 48, lon: 84, time: 20)
Coordinates:
  * lat      (lat) float64 384B 41.62 41.67 41.72 41.77 ... 43.87 43.92 43.97
  * lon      (lon) float64 672B -9.575 -9.525 -9.475 ... -5.525 -5.475 -5.425
  * time     (time) datetime64[ns] 160B 1995-01-01 1996-01-01 ... 2014-01-01
    height   float64 8B 2.0
Data variables:
    dtr      (time, lat, lon) float32 323kB nan nan nan nan ... nan nan nan nan
Attributes:
    earthkit_provenance:  {'indicator_definition': {'tasmin': Parameter(kind=...